In [1]:
"""
Customer Segmentation Analysis - Part 3: Market Basket & Recommendations
========================================================================

This notebook provides:
1. Market basket analysis and association rules
2. Segment-specific product recommendations
3. Cross-selling and upselling opportunities
4. Final business recommendations
"""

'\nCustomer Segmentation Analysis - Part 3: Market Basket & Recommendations\n========================================================================\n\nThis notebook provides:\n1. Market basket analysis and association rules\n2. Segment-specific product recommendations\n3. Cross-selling and upselling opportunities\n4. Final business recommendations\n'

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

In [5]:
# Load data
print("Loading data...")
df_transactions = pd.read_csv('TOKEN.csv').merge(
    pd.read_csv('VENTE.csv'), on='CD_TICKET_UNIQUE'
).merge(
    pd.read_csv('ARTICLE.csv'), on='ID_ARTICLE', how='left'
)

customer_segments = pd.read_csv('customers_with_segments.csv')

print(f"✓ Transaction data loaded: {df_transactions.shape}")
print(f"✓ Customer segments loaded: {customer_segments.shape}")

Loading data...
✓ Transaction data loaded: (1305191, 18)
✓ Customer segments loaded: (121112, 28)


In [6]:
# ============================================================================
# PART 7: MARKET BASKET ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("PART 7: MARKET BASKET ANALYSIS")
print("=" * 80)


PART 7: MARKET BASKET ANALYSIS


In [7]:
# ============================================================================
# STEP 7.1: Prepare Transaction Baskets
# ============================================================================

print("\n[STEP 7.1] Preparing transaction baskets...")

# Create baskets at family level (more manageable than individual products)
baskets = df_transactions.groupby(['CD_TICKET_UNIQUE', 'LB_FAMILLE'])['QT_UVC'].sum().reset_index()
baskets = baskets[baskets['LB_FAMILLE'].notna()]

# Create basket format for association rules
basket_items = baskets.groupby('CD_TICKET_UNIQUE')['LB_FAMILLE'].apply(list).reset_index()

print(f"✓ Created {len(basket_items)} baskets")
print(f"  Average items per basket: {baskets.groupby('CD_TICKET_UNIQUE').size().mean():.2f}")

# Sample of baskets
print("\nSample baskets:")
for i in range(min(5, len(basket_items))):
    print(f"  Basket {i+1}: {basket_items.iloc[i]['LB_FAMILLE'][:5]}")



[STEP 7.1] Preparing transaction baskets...
✓ Created 174007 baskets
  Average items per basket: 6.32

Sample baskets:
  Basket 1: ['Agrumes', 'Banane', 'Prêt à déguster', 'Pâte fraîche']
  Basket 2: ['Agrumes', 'Avocat', 'Banane', 'Courge', 'Courgette']
  Basket 3: ['Plat familial']
  Basket 4: ['Banane', 'Pomme']
  Basket 5: ['Oeufs', 'Poissons']


In [8]:
# ============================================================================
# STEP 7.2: Generate Association Rules - Overall
# ============================================================================

print("\n[STEP 7.2] Generating association rules (overall)...")

# Convert to one-hot encoded format
te = TransactionEncoder()
te_ary = te.fit(basket_items['LB_FAMILLE']).transform(basket_items['LB_FAMILLE'])
basket_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"✓ One-hot encoded baskets: {basket_encoded.shape}")

# Find frequent itemsets
print("  Finding frequent itemsets...")
frequent_itemsets = apriori(basket_encoded, min_support=0.01, use_colnames=True)
print(f"✓ Found {len(frequent_itemsets)} frequent itemsets")

# Generate association rules
print("  Generating association rules...")
if len(frequent_itemsets) > 0:
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
    rules = rules.sort_values('lift', ascending=False)
    
    print(f"✓ Generated {len(rules)} association rules")
    
    # Display top rules
    print("\nTop 10 Association Rules (by Lift):")
    print("=" * 120)
    top_rules = rules.head(10)[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    
    for idx, row in top_rules.iterrows():
        ant = list(row['antecedents'])[0] if len(row['antecedents']) == 1 else row['antecedents']
        cons = list(row['consequents'])[0] if len(row['consequents']) == 1 else row['consequents']
        print(f"\n{ant}")
        print(f"  → {cons}")
        print(f"  Support: {row['support']:.3f} | Confidence: {row['confidence']:.3f} | Lift: {row['lift']:.2f}")
    
    # Save rules
    rules.to_csv('association_rules_overall.csv', index=False)
    print("\n✓ Association rules saved to 'association_rules_overall.csv'")
else:
    print("⚠ No frequent itemsets found. Consider lowering min_support threshold.")
    rules = pd.DataFrame()



[STEP 7.2] Generating association rules (overall)...
✓ One-hot encoded baskets: (174007, 120)
  Finding frequent itemsets...
✓ Found 2226 frequent itemsets
  Generating association rules...
✓ Generated 12222 association rules

Top 10 Association Rules (by Lift):

frozenset({'Carotte', 'Poireau'})
  → frozenset({'Navet et Rave', 'Agrumes'})
  Support: 0.011 | Confidence: 0.240 | Lift: 8.36

frozenset({'Navet et Rave', 'Agrumes'})
  → frozenset({'Carotte', 'Poireau'})
  Support: 0.011 | Confidence: 0.370 | Lift: 8.36

Navet et Rave
  → frozenset({'Carotte', 'Poireau', 'Agrumes'})
  Support: 0.011 | Confidence: 0.238 | Lift: 7.58

frozenset({'Carotte', 'Poireau', 'Agrumes'})
  → Navet et Rave
  Support: 0.011 | Confidence: 0.339 | Lift: 7.58

frozenset({'Carotte', 'Poireau'})
  → Navet et Rave
  Support: 0.015 | Confidence: 0.332 | Lift: 7.43

Navet et Rave
  → frozenset({'Carotte', 'Poireau'})
  Support: 0.015 | Confidence: 0.329 | Lift: 7.43

frozenset({'Carotte', 'Navet et Rave'})
  →

In [9]:
# ============================================================================
# STEP 7.3: Segment-Specific Association Rules
# ============================================================================

print("\n[STEP 7.3] Generating segment-specific association rules...")

# Merge segments with transactions
df_trans_segments = df_transactions.merge(
    customer_segments[['NO_TOKEN_CB', 'Cluster']], 
    on='NO_TOKEN_CB', 
    how='left'
)

# Generate rules for each segment
segment_rules_dict = {}

for cluster_id in sorted(customer_segments['Cluster'].unique()):
    print(f"\n  Analyzing Segment {cluster_id}...")
    
    # Get transactions for this segment
    segment_trans = df_trans_segments[df_trans_segments['Cluster'] == cluster_id]
    
    # Create baskets
    segment_baskets = segment_trans.groupby(['CD_TICKET_UNIQUE', 'LB_FAMILLE'])['QT_UVC'].sum().reset_index()
    segment_baskets = segment_baskets[segment_baskets['LB_FAMILLE'].notna()]
    
    if len(segment_baskets) < 10:
        print(f"    ⚠ Insufficient data for Segment {cluster_id}")
        continue
    
    segment_basket_items = segment_baskets.groupby('CD_TICKET_UNIQUE')['LB_FAMILLE'].apply(list).reset_index()
    
    # Encode and find patterns
    try:
        te_seg = TransactionEncoder()
        te_seg_ary = te_seg.fit(segment_basket_items['LB_FAMILLE']).transform(segment_basket_items['LB_FAMILLE'])
        basket_seg_encoded = pd.DataFrame(te_seg_ary, columns=te_seg.columns_)
        
        freq_items_seg = apriori(basket_seg_encoded, min_support=0.02, use_colnames=True)
        
        if len(freq_items_seg) > 0:
            rules_seg = association_rules(freq_items_seg, metric="lift", min_threshold=1.0)
            rules_seg = rules_seg.sort_values('lift', ascending=False)
            segment_rules_dict[cluster_id] = rules_seg
            
            print(f"    ✓ Found {len(rules_seg)} rules for Segment {cluster_id}")
            
            # Save segment rules
            rules_seg.to_csv(f'association_rules_segment_{cluster_id}.csv', index=False)
        else:
            print(f"    ⚠ No rules found for Segment {cluster_id}")
    except Exception as e:
        print(f"    ⚠ Error processing Segment {cluster_id}: {str(e)}")



[STEP 7.3] Generating segment-specific association rules...

  Analyzing Segment 0...
    ✓ Found 278 rules for Segment 0

  Analyzing Segment 1...
    ✓ Found 860 rules for Segment 1

  Analyzing Segment 2...
    ✓ Found 173558 rules for Segment 2

  Analyzing Segment 3...
    ✓ Found 440 rules for Segment 3


In [10]:
# ============================================================================
# PART 8: PRODUCT RECOMMENDATIONS BY SEGMENT
# ============================================================================

print("\n" + "=" * 80)
print("PART 8: PRODUCT RECOMMENDATIONS BY SEGMENT")
print("=" * 80)



PART 8: PRODUCT RECOMMENDATIONS BY SEGMENT


In [11]:
# ============================================================================
# STEP 8.1: Top Products by Segment
# ============================================================================

print("\n[STEP 8.1] Identifying top products by segment...")

# Get top product families by segment
top_products_by_segment = df_trans_segments.groupby(['Cluster', 'LB_FAMILLE']).agg({
    'QT_UVC': 'sum',
    'MT_TTC_NET': 'sum',
    'CD_TICKET_UNIQUE': 'nunique'
}).reset_index()

top_products_by_segment.columns = ['Cluster', 'Product_Family', 'Total_Quantity', 'Total_Revenue', 'Unique_Transactions']
top_products_by_segment = top_products_by_segment.sort_values(['Cluster', 'Total_Revenue'], ascending=[True, False])

print("\nTop 5 Product Families by Segment:")
print("=" * 100)

for cluster_id in sorted(customer_segments['Cluster'].unique()):
    segment_products = top_products_by_segment[top_products_by_segment['Cluster'] == cluster_id].head(5)
    print(f"\n📊 Segment {cluster_id}:")
    for idx, row in segment_products.iterrows():
        print(f"  {row['Product_Family']}: €{row['Total_Revenue']:,.0f} revenue | {row['Unique_Transactions']:,} transactions")

# Save
top_products_by_segment.to_csv('top_products_by_segment.csv', index=False)
print("\n✓ Top products saved to 'top_products_by_segment.csv'")



[STEP 8.1] Identifying top products by segment...

Top 5 Product Families by Segment:

📊 Segment 0:
  Agrumes: €74,487 revenue | 13,089 transactions
  Légumes Exotiques: €31,216 revenue | 4,854 transactions
  Fruits Exotiques: €29,423 revenue | 5,072 transactions
  Condiment: €19,556 revenue | 6,131 transactions
  Pomme De Terre: €17,801 revenue | 4,892 transactions

📊 Segment 1:
  Agrumes: €174,139 revenue | 35,105 transactions
  Poissons: €114,779 revenue | 11,753 transactions
  Crustacés: €66,854 revenue | 7,984 transactions
  Fromage à consommer chaud: €55,504 revenue | 4,152 transactions
  Coquillages: €50,730 revenue | 3,982 transactions

📊 Segment 2:
  Agrumes: €118,426 revenue | 15,679 transactions
  Poissons: €108,600 revenue | 7,178 transactions
  Crustacés: €68,178 revenue | 5,424 transactions
  Coquillages: €63,874 revenue | 3,274 transactions
  Poisson fumé: €55,913 revenue | 4,388 transactions

📊 Segment 3:
  Agrumes: €77,046 revenue | 15,894 transactions
  Poissons: €42

In [12]:
# ============================================================================
# STEP 8.2: Cross-Selling Opportunities
# ============================================================================

print("\n[STEP 8.2] Identifying cross-selling opportunities...")

# For each segment, find products frequently bought together
cross_sell_opportunities = []

for cluster_id, rules_df in segment_rules_dict.items():
    if len(rules_df) > 0:
        # Get top cross-sell rules
        top_cross_sell = rules_df.nlargest(5, 'lift')
        
        for idx, rule in top_cross_sell.iterrows():
            cross_sell_opportunities.append({
                'Segment': cluster_id,
                'If_Customer_Buys': list(rule['antecedents']),
                'Recommend': list(rule['consequents']),
                'Confidence': rule['confidence'],
                'Lift': rule['lift']
            })

cross_sell_df = pd.DataFrame(cross_sell_opportunities)

if len(cross_sell_df) > 0:
    print("\nTop Cross-Selling Opportunities by Segment:")
    print("=" * 100)
    for cluster_id in cross_sell_df['Segment'].unique():
        segment_opps = cross_sell_df[cross_sell_df['Segment'] == cluster_id]
        print(f"\n🎯 Segment {cluster_id}:")
        for idx, row in segment_opps.iterrows():
            print(f"  If buys: {row['If_Customer_Buys']}")
            print(f"    → Recommend: {row['Recommend']} (Lift: {row['Lift']:.2f}, Confidence: {row['Confidence']:.1%})")
    
    cross_sell_df.to_csv('cross_sell_opportunities.csv', index=False)
    print("\n✓ Cross-sell opportunities saved to 'cross_sell_opportunities.csv'")
else:
    print("⚠ No cross-sell opportunities identified")



[STEP 8.2] Identifying cross-selling opportunities...

Top Cross-Selling Opportunities by Segment:

🎯 Segment 0:
  If buys: ['Navet et Rave']
    → Recommend: ['Carotte'] (Lift: 5.34, Confidence: 60.0%)
  If buys: ['Carotte']
    → Recommend: ['Navet et Rave'] (Lift: 5.34, Confidence: 18.3%)
  If buys: ['Poireau']
    → Recommend: ['Carotte'] (Lift: 4.48, Confidence: 50.4%)
  If buys: ['Carotte']
    → Recommend: ['Poireau'] (Lift: 4.48, Confidence: 19.6%)
  If buys: ['Carotte']
    → Recommend: ['Courgette'] (Lift: 4.28, Confidence: 23.3%)

🎯 Segment 1:
  If buys: ['Carotte']
    → Recommend: ['Navet et Rave'] (Lift: 4.23, Confidence: 16.5%)
  If buys: ['Navet et Rave']
    → Recommend: ['Carotte'] (Lift: 4.23, Confidence: 63.4%)
  If buys: ['Carotte', 'Agrumes']
    → Recommend: ['Poireau'] (Lift: 3.73, Confidence: 28.5%)
  If buys: ['Poireau']
    → Recommend: ['Carotte', 'Agrumes'] (Lift: 3.73, Confidence: 33.4%)
  If buys: ['Poireau', 'Agrumes']
    → Recommend: ['Carotte'] (Lift

In [18]:
# ============================================================================
# FINAL OUTPUT SUMMARY
# ============================================================================

print("\n" + "=" * 100)
print("ANALYSIS COMPLETE!")
print("=" * 100)

print("\n📊 Generated Outputs:")
outputs = [
    "1. customer_features_prepared.csv - Customer features for clustering",
    "2. customers_with_segments.csv - All customers with assigned segments",
    "3. segment_profiles.csv - Statistical profiles of each segment",
    "4. segment_business_summary.csv - Business metrics by segment",
    "5. association_rules_overall.csv - Overall association rules",
    "6. association_rules_segment_X.csv - Segment-specific rules",
    "7. top_products_by_segment.csv - Best-selling products per segment",
    "8. cross_sell_opportunities.csv - Cross-selling recommendations",
    "9. store_segment_distribution.csv - Segment mix by store",
    "10. store_performance_by_segment.csv - Store-segment performance",
    "11. executive_summary.txt - Executive summary report",
    "12. Various visualization PNG files"
]

for output in outputs:
    print(f"  {output}")

print("\n🎯 Ready for Business Implementation!")
print("=" * 100)
























ANALYSIS COMPLETE!

📊 Generated Outputs:
  1. customer_features_prepared.csv - Customer features for clustering
  2. customers_with_segments.csv - All customers with assigned segments
  3. segment_profiles.csv - Statistical profiles of each segment
  4. segment_business_summary.csv - Business metrics by segment
  5. association_rules_overall.csv - Overall association rules
  6. association_rules_segment_X.csv - Segment-specific rules
  7. top_products_by_segment.csv - Best-selling products per segment
  8. cross_sell_opportunities.csv - Cross-selling recommendations
  9. store_segment_distribution.csv - Segment mix by store
  10. store_performance_by_segment.csv - Store-segment performance
  11. executive_summary.txt - Executive summary report
  12. Various visualization PNG files

🎯 Ready for Business Implementation!
